# Model Context Protocol (MCP) & LangGraph Resume Matching Agent

This notebook demonstrates the end-to-end implementation and execution of a **Resume Matching & Ranking Agent** using **LangGraph** and the **Model Context Protocol (MCP)**. 

### System Architecture:
1. **Filesystem MCP Server** (`mcp_servers/filesystem_mcp_server.py`): Exposes sandboxed file management, directory polling, and parallel PDF text extraction.
2. **Candidate Database MCP Server** (`mcp_servers/db_mcp_server.py`): Exposes candidate profile searches (salary, certifications, background checks) representing HR databases.
3. **LangGraph Agent** (`agent/matching_agent.py`): Coordinates the workflow over stdio MCP client connections to match candidate PDF resumes to a job description.

Let's get started!

## 1. Environment & Setup

First, we load environment variables from the `.env` file and verify our Python environment and dependencies.

In [ ]:
import os
import sys
import asyncio
import json
from dotenv import load_dotenv

# Load OpenRouter keys and configuration
load_dotenv()

print(f"Python interpreter: {sys.executable}")
print(f"Project workspace: {os.getcwd()}")
print(f"OpenRouter API key loaded: {'Yes' if os.getenv('OPENROUTER_API_KEY') else 'No (using keyword mock fallback)'}")

## 2. Check Available Resumes

We verify that the persistent resumes directory exists and list the PDF files available for matching.

In [ ]:
import os
RESUMES_DIR = './resumes'
REPORTS_DIR = './reports'
os.makedirs(REPORTS_DIR, exist_ok=True)
print('Found persistent resumes:', os.listdir(RESUMES_DIR))

## 3. Launch and Connect to MCP Servers

Using the official `mcp` client libraries, we start both local servers (`filesystem_mcp_server.py` and `db_mcp_server.py`) as subprocesses and initialize sessions.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Launch configurations pointing to modular scripts
fs_server_params = StdioServerParameters(
    command=sys.executable,
    args=["mcp_servers/filesystem_mcp_server.py"]
)

db_server_params = StdioServerParameters(
    command=sys.executable,
    args=["mcp_servers/db_mcp_server.py"]
)

print("Connecting to Filesystem and Database MCP servers...")

## 4. Discover Server Capabilities (Tools & Resources)

Let's handshake with the servers and query their lists of tools and resources.

In [ ]:
async with stdio_client(fs_server_params) as (fs_read, fs_write), \
           stdio_client(db_server_params) as (db_read, db_write):
           
    async with ClientSession(fs_read, fs_write) as fs_session, \
               ClientSession(db_read, db_write) as db_session:
               
        # Protocol Handshake
        await fs_session.initialize()
        await db_session.initialize()
        
        print("--- Filesystem MCP Server Tools ---")
        fs_tools = await fs_session.list_tools()
        for t in fs_tools.tools:
            print(f"- {t.name}: {t.description}")
            
        print("\n--- Filesystem MCP Server Resources ---")
        fs_resources = await fs_session.list_resources()
        for r in fs_resources.resources:
            print(f"- {r.uri}: {r.description}")

        print("\n--- Database MCP Server Tools ---")
        db_tools = await db_session.list_tools()
        for t in db_tools.tools:
            print(f"- {t.name}: {t.description}")

## 5. Direct Execution of Server Tools

Let's verify individual capabilities: PDF text extraction, sandbox boundaries, and directory watching.

In [ ]:
from src.agent.matching_agent import parse_mcp_result

async with stdio_client(fs_server_params) as (fs_read, fs_write):
    async with ClientSession(fs_read, fs_write) as fs_session:
        await fs_session.initialize()
        
        # 1. Read & parse a PDF file programmatically
        res = await fs_session.call_tool("read_file", arguments={"path": "./resumes/Alice_Smith_Resume.pdf"})
        content = parse_mcp_result(res)
        print("--- parsed PDF Resume snippet ---")
        print(content[:250].encode('ascii', errors='ignore').decode('ascii'))
        
        # 2. Test sandbox protection boundary
        print("\n--- testing sandboxed paths boundary ---")
        sandbox_res = await fs_session.call_tool("list_directory", arguments={"path": "../../"})
        print(parse_mcp_result(sandbox_res))

## 6. Run LangGraph Matching Workflow

We will now execute the matching workflow using our modular `matching_agent`. This runs the LangGraph state machine: reading the resumes in parallel (`batch_process`), fetching candidate background info (`db_server`), scoring them, ranking them, and writing `latest_report.md` to the disk.

In [ ]:
from src.agent.matching_agent import run_matching_workflow

JOB_DESCRIPTION = """
Looking for a Senior Python Developer with extensive experience in LangChain, LangGraph, and cloud systems (AWS).
Required Skills: Python, LangGraph, LangChain, AWS, Docker.
Nice to have: SQL, Javascript.
"""

# Execute LangGraph matching workflow
report = await run_matching_workflow(JOB_DESCRIPTION, RESUMES_DIR)

print("\n================== PIPELINE REPORT CONTENT ==================")
print(report)
print("=============================================================")